# Tutorial: Intelligence Pipeline Basics

Audience:
- Developers running the full intelligence pipeline end-to-end.

Prerequisites:
- Working project environment with provider access.

Learning goals:
- Configure intelligence settings.
- Run the pipeline for a symbol set.
- Inspect opportunities, themes, and stored state.


## Outline

1. Setup imports and configuration
2. Run the intelligence pipeline
3. Review opportunities and theme clusters
4. Load persisted outputs from storage


In [1]:
from datetime import datetime

from swing_screener.intelligence import (
    CatalystConfig,
    IntelligenceConfig,
    IntelligenceStorage,
    OpportunityConfig,
    run_intelligence_pipeline,
)


## Step 1 - Build a baseline config and symbol universe

This mirrors the original script defaults for providers, catalyst settings, and opportunity weighting.


In [4]:
cfg = IntelligenceConfig(
    enabled=True,
    providers=("yahoo_finance",),
    catalyst=CatalystConfig(
        lookback_hours=240,
        false_catalyst_return_z=1.5,
    ),
    opportunity=OpportunityConfig(
        technical_weight=0.55,
        catalyst_weight=0.45,
        max_daily_opportunities=10,
        min_opportunity_score=0.5,
    ),
)

symbols = ["NVDA", "AMD", "INTC", "TSM", "AVGO", "QCOM", "MRVL", "TXN"]
print(f"Running intelligence pipeline for {len(symbols)} symbols...")
print(f"Config: providers={cfg.providers}, lookback={cfg.catalyst.lookback_hours}h")


Running intelligence pipeline for 8 symbols...
Config: providers=('yahoo_finance',), lookback=240h


## Step 2 - Run the pipeline snapshot

Run once at `datetime.now()` and inspect high-level counts.


In [5]:
snapshot = run_intelligence_pipeline(
    symbols=symbols,
    cfg=cfg,
    asof_dt=datetime.now(),
)

print(f"Results for {snapshot.asof_date}:")
print(f"  Events collected: {len(snapshot.events)}")
print(f"  Signals generated: {len(snapshot.signals)}")
print(f"  Themes detected: {len(snapshot.themes)}")
print(f"  Opportunities: {len(snapshot.opportunities)}")


Results for 2026-02-24:
  Events collected: 80
  Signals generated: 80
  Themes detected: 0
  Opportunities: 0


/Users/matteo.longo/projects/randomness/trading/swing_screener/.venv/lib/python3.11/site-packages/portalocker/utils.py:231: UserWarning: timeout has no effect in blocking mode
  warnings.warn(


## Step 3 - Inspect top opportunities and detected themes

Limit prints to a few rows to keep notebook output readable.


In [6]:
if snapshot.opportunities:
    print("Top Opportunities:")
    for opp in snapshot.opportunities[:5]:
        print(
            f"  {opp.symbol}: score={opp.opportunity_score:.3f}, "
            f"technical={opp.technical_readiness:.2f}, "
            f"catalyst={opp.catalyst_strength:.2f}, "
            f"state={opp.state}"
        )

if snapshot.themes:
    print("Theme Clusters:")
    for theme in snapshot.themes:
        print(
            f"  {theme.name}: {theme.symbols} (strength={theme.cluster_strength:.2f})"
        )


## Step 4 - Load persisted state from storage

The pipeline stores outputs; here we verify by loading opportunities and symbol states.


In [7]:
storage = IntelligenceStorage()
loaded_opps = storage.load_opportunities(snapshot.asof_date)
print(f"Loaded {len(loaded_opps)} opportunities from storage")

states = storage.load_symbol_state()
print(f"Current symbol states: {len(states)} symbols")
for symbol, state in list(states.items())[:5]:
    print(f"  {symbol}: {state.state} (score={state.state_score:.2f})")


Loaded 0 opportunities from storage
Current symbol states: 8 symbols
  AMD: QUIET (score=0.00)
  AVGO: QUIET (score=0.00)
  INTC: QUIET (score=0.00)
  MRVL: QUIET (score=0.00)
  NVDA: QUIET (score=0.00)
